# Bài thực hành 4 – TV1: Tích hợp và tổng hợp kết quả
**Wavelet Hash + khoảng cách Hamming** – chạy toàn bộ luồng từ ảnh đầu vào đến kết quả so khớp, tổng hợp bảng so sánh.

```
ảnh ──► wavelet_hash.py (TV4) ──► common.normalize_hash ──► similarity.py (TV5) ──► ngưỡng ──► evaluation.py (TV6)
          hex 64 bit                bit 0/1                  khoảng cách Hamming      d <= t     Acc/Recall/Spec/Prec, ROC, AUC
                                                                                                     └─► experiments.py (TV6): so sánh cấu hình
```
Quy ước: `label` 1 = tương tự · `distance` càng nhỏ càng giống · dự đoán `distance <= threshold` · ROC dùng điểm số `-distance`.

## 1. Kiểm tra tích hợp (import, hàm cần có, đường dẫn)

In [1]:
import os, sys
for p in (".", "src", "../src", "lab_4/src"):
    if os.path.exists(os.path.join(p, "common.py")):
        sys.path.insert(0, os.path.abspath(p)); break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import common, main

pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)
status = common.module_status()
status

,tên,loại,ok,chi tiết
0,numpy,thư viện,True,2.4.6
1,pandas,thư viện,True,3.0.3
2,cv2,thư viện,True,4.13.0
3,pywt,thư viện,True,1.8.0
4,matplotlib,thư viện,True,3.11.0
5,wavelet_hash,module nhóm,False,ImportError: cannot import name 'to_grayscale' from 'common' (c:\Users\LENOVO\OneDrive...
6,similarity,module nhóm,True,C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\src\similarity.py
7,evaluation,module nhóm,True,C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\src\evaluation.py
8,experiments,module nhóm,True,C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\src\experiments.py


In [2]:
common.path_status()

,mục,đường dẫn,tồn tại,ghi chú
0,thư mục ảnh,C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\data\input,True,15 ảnh
1,pairs.csv,C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\data\pairs.csv,False,tuỳ chọn - nếu không có sẽ tự tạo cặp từ thư mục ảnh


Nếu cột `ok` có `False`: xem cột `chi tiết` và mục *Xử lý lỗi thường gặp* trong `HUONG_DAN_CHAY.md`
(thường là import sai, hai file cùng tên, hoặc chưa restart kernel sau khi thay file).

## 2. Cấu hình chạy
Thứ tự ưu tiên dữ liệu: `pairs.csv` → tự tạo cặp từ `data/input` → ảnh giả lập (chỉ để minh hoạ, **không dùng để báo cáo**).

In [3]:
DATA_DIR   = common.DATA_DIR
PAIRS_FILE = common.PAIRS_FILE
OUT_DIR    = common.OUTPUT_DIR
THRESHOLD  = None        # None = tự chọn theo Youden J; hoặc đặt số nguyên, ví dụ 10

n_imgs = len([f for f in os.listdir(DATA_DIR) if f.lower().endswith(common.IMAGE_EXTS)]) if os.path.isdir(DATA_DIR) else 0
DEMO = not os.path.isfile(PAIRS_FILE) and n_imgs < 2
if DEMO:
    print("⚠ Không có pairs.csv và < 2 ảnh trong", DATA_DIR, "-> dùng ảnh GIẢ LẬP (chỉ để minh hoạ).")
else:
    print("Dữ liệu thật:", "pairs.csv" if os.path.isfile(PAIRS_FILE) else f"{n_imgs} ảnh trong {DATA_DIR}")

Dữ liệu thật: 15 ảnh trong C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\data\input


## 3. Chạy toàn bộ pipeline

In [4]:
res = main.run_pipeline(DATA_DIR, PAIRS_FILE, OUT_DIR, threshold=THRESHOLD, compare=True, demo=DEMO)

[1/5] Dữ liệu: tự tạo cặp từ 15 ảnh trong C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\data\input | 210 cặp (105 tương tự / 105 khác nhau)
[2/5] Hash: 120/120 ảnh OK | nguồn hash: experiments.py (dự phòng: haar/1 mức/median)
[3/5] Khoảng cách: 210/210 cặp hợp lệ | trạng thái: {'ok': 210}
[4/5] Ngưỡng tự chọn (Youden J): 14
[5/5] Đánh giá tại ngưỡng 14: Acc=0.995 Recall=1.000 Spec=0.990 Prec=0.991 | AUC=1.000
[so sánh] 24 cấu hình | tốt nhất: db2|L1|mean (AUC=1.000)
Đã ghi kết quả vào: C:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\data\output


In [5]:
tbl = res["table"]
print(res["summary"]["status_counts"])
tbl[["img1", "img2", "label", "distance", "status", "prediction"]].head(10)

{'ok': 210}


,img1,img2,label,distance,status,prediction
0,anh_dem,anh_dem::darken,1,0,ok,1.0
1,anh_dem,anh_dem::brighten,1,0,ok,1.0
2,anh_dem,anh_dem::gauss_noise,1,0,ok,1.0
3,anh_dem,anh_dem::blur,1,0,ok,1.0
4,anh_dem,anh_dem::jpeg_q30,1,0,ok,1.0
5,anh_dem,anh_dem::rotate_5deg,1,0,ok,1.0
6,anh_dem,anh_dem::crop_90pct,1,6,ok,1.0
7,buom,buom::darken,1,0,ok,1.0
8,buom,buom::brighten,1,0,ok,1.0
9,buom,buom::gauss_noise,1,0,ok,1.0


## 4. Chỉ số theo ngưỡng và đường cong ROC

In [6]:
thr = res["threshold"]
sm = res["sweep_metrics"]
sm[sm.threshold.isin(sorted(set(range(0, 33, 4)) | {thr}))].round(3)

,threshold,comparison,TP,FP,TN,FN,accuracy,recall,specificity,precision
0,0,<=,68,0,105,37,0.824,0.648,1.000,1.000
4,4,<=,87,0,105,18,0.914,0.829,1.000,1.000
8,8,<=,98,0,105,7,0.967,0.933,1.000,1.000
12,12,<=,104,1,104,1,0.990,0.990,0.990,0.990
14,14,<=,105,1,104,0,0.995,1.000,0.990,0.991
16,16,<=,105,9,96,0,0.957,1.000,0.914,0.921
20,20,<=,105,14,91,0,0.933,1.000,0.867,0.882
24,24,<=,105,19,86,0,0.910,1.000,0.819,0.847
28,28,<=,105,33,72,0,0.843,1.000,0.686,0.761
32,32,<=,105,66,39,0,0.686,1.000,0.371,0.614


In [7]:
m, roc = res["metrics"], res["roc"]
fpr_pt = m["FP"] / (m["FP"] + m["TN"])
plt.figure(figsize=(5.5, 5.5))
plt.plot(roc["fpr"], roc["tpr"], drawstyle="steps-post", label=f"AUC = {roc['auc']:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Ngẫu nhiên")
plt.scatter([fpr_pt], [m["recall"]], c="red", zorder=3, label=f"Ngưỡng {thr}")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC – pipeline tích hợp")
plt.legend(loc="lower right"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print(f"Ngưỡng {thr}: Accuracy={m['accuracy']:.3f}  Recall={m['recall']:.3f}  "
      f"Specificity={m['specificity']:.3f}  Precision={m['precision']:.3f}  AUC={roc['auc']:.3f}")

Ngưỡng 14: Accuracy=0.995  Recall=1.000  Specificity=0.990  Precision=0.991  AUC=1.000


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_3780\3430419092.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(loc="lower right"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()


## 5. Bảng so sánh tổng hợp
Gộp kết quả **pipeline chính** (`wavelet_hash.py` của TV4) với các cấu hình trong `experiments.py` (wavelet × mức phân rã × lượng tử hoá).
Lưu ý: các cấu hình thử nghiệm dùng bộ dữ liệu/ảnh thu nhỏ của `experiments.py`, nên AUC có thể lệch nhẹ so với pipeline.

In [8]:
pipe_row = dict(config="PIPELINE (wavelet_hash.py)", wavelet="haar", level=1, quant="median",
                auc=roc["auc"], threshold=thr, accuracy=m["accuracy"], recall=m["recall"],
                specificity=m["specificity"], precision=m["precision"])
cmp_df = res["comparison"]
final = pd.concat([pd.DataFrame([pipe_row]), cmp_df], ignore_index=True) if cmp_df is not None else pd.DataFrame([pipe_row])
final = final.sort_values(["auc", "accuracy"], ascending=False).reset_index(drop=True)
final.to_csv(os.path.join(OUT_DIR, "comparison_table.csv"), index=False)
final.round(3)

,config,wavelet,level,quant,auc,threshold,accuracy,recall,specificity,precision
0,db2|L1|mean,db2,1,mean,1.000,12,1.000,1.000,1.000,1.000
1,sym2|L1|mean,sym2,1,mean,1.000,12,1.000,1.000,1.000,1.000
2,coif1|L1|median,coif1,1,median,1.000,10,1.000,1.000,1.000,1.000
3,coif1|L1|mean,coif1,1,mean,1.000,11,1.000,1.000,1.000,1.000
4,coif1|L2|mean,coif1,2,mean,1.000,12,1.000,1.000,1.000,1.000
5,coif1|L3|median,coif1,3,median,1.000,12,1.000,1.000,1.000,1.000
6,coif1|L3|mean,coif1,3,mean,1.000,12,1.000,1.000,1.000,1.000
7,coif1|L2|median,coif1,2,median,1.000,14,0.995,1.000,0.990,0.991
8,db2|L1|median,db2,1,median,1.000,12,0.995,1.000,0.990,0.991
9,sym2|L1|median,sym2,1,median,1.000,12,0.995,1.000,0.990,0.991


In [9]:
top = final.head(10).iloc[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh(top.config, top.auc)
plt.xlabel("AUC"); plt.title("10 cấu hình có AUC cao nhất"); plt.grid(alpha=.3, axis="x")
plt.xlim(max(0, top.auc.min() - 0.05), 1.01); plt.tight_layout(); plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_3780\1663729656.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlim(max(0, top.auc.min() - 0.05), 1.01); plt.tight_layout(); plt.show()


## 6. Các cặp phân loại sai của pipeline

In [10]:
ok = tbl[tbl.status == "ok"].copy()
ok["prediction"] = ok.prediction.astype(int)
fn = ok[(ok.label == 1) & (ok.prediction == 0)]
fp = ok[(ok.label == 0) & (ok.prediction == 1)]
print(f"Bỏ sót (FN): {len(fn)} | Nhầm là giống (FP): {len(fp)} | Cặp lỗi bị loại: {int((tbl.status != 'ok').sum())}")
errs = pd.concat([fn.assign(loai="FN"), fp.assign(loai="FP")])
errs[["loai", "img1", "img2", "label", "distance"]].head(15) if len(errs) else "Không có cặp sai ở ngưỡng này."

Bỏ sót (FN): 0 | Nhầm là giống (FP): 1 | Cặp lỗi bị loại: 0


,loai,img1,img2,label,distance
114,FP,suong_mu,song_bien,0,10


## 7. Checklist tích hợp tự động

In [11]:
s = res["summary"]
checks = [
    ("Đủ thư viện (numpy, pandas, cv2, pywt, matplotlib)", bool(status[status["loại"] == "thư viện"].ok.all())),
    ("Có wavelet_hash.py của TV4 (không dùng hàm dự phòng)", s["hash_source"].startswith("wavelet_hash.py")),
    ("similarity.py + evaluation.py import được", bool(status[status["tên"].isin(["similarity", "evaluation"])].ok.all())),
    ("experiments.py đủ hàm (so sánh cấu hình chạy được)", bool(status.set_index("tên").loc["experiments", "ok"]) and res["comparison"] is not None),
    ("Mọi ảnh hash thành công", s["n_hash_ok"] == s["n_images"]),
    ("Mọi cặp hợp lệ (không missing/invalid/length_mismatch)", s["n_pairs_ok"] == s["n_pairs"]),
    ("Hash đều 64 bit", s["n_bits"] == 64),
    ("Quy ước so sánh thống nhất: distance <= threshold", s["comparison"] == "distance <= threshold"),
    ("Có đủ file kết quả trong output", all(os.path.exists(os.path.join(OUT_DIR, f)) for f in
        ["distance_predictions.csv", "threshold_metrics.csv", "roc.png", "summary.json", "comparison_table.csv"])),
    ("Dùng dữ liệu thật (không phải ảnh giả lập)", not DEMO),
]
pd.DataFrame(checks, columns=["Hạng mục", "Đạt"])

,Hạng mục,Đạt
0,"Đủ thư viện (numpy, pandas, cv2, pywt, matplotlib)",True
1,Có wavelet_hash.py của TV4 (không dùng hàm dự phòng),False
2,similarity.py + evaluation.py import được,True
3,experiments.py đủ hàm (so sánh cấu hình chạy được),True
4,Mọi ảnh hash thành công,True
5,Mọi cặp hợp lệ (không missing/invalid/length_mismatch),True
6,Hash đều 64 bit,True
7,Quy ước so sánh thống nhất: distance <= threshold,True
8,Có đủ file kết quả trong output,True
9,Dùng dữ liệu thật (không phải ảnh giả lập),True
